<a href="https://colab.research.google.com/github/aryalankit121/CIS-3400-Advanced-Programming/blob/main/CIS3400_Laundromat_Reusable_Classes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CIS 3400 — Reusable Python Classes and CSV Persistence
### Guided Laundromat Lab
### Your Name: Ankit Aryal
### Date: 11/20/2026
### At end of lab, please submit your .ipynb file AND the .ipynb file saved "printed" as a PDF. Optional: add to your class github repository.

**Learning goals:** Save a class in a `.py` module, import it into two different applications, and persist order records in a CSV file.

**Run cells in order.** This notebook works in Jupyter or Google Colab. All files are created in the current working folder. In Colab, files in the temporary runtime may disappear when the runtime is reset; download them or use Drive for long-term storage.

## Part 1 — Build a reusable class (10 minutes)

We will save a real Python source file named `laundry_order.py`. The class represents **one order**. It calculates a total and can convert itself to a dictionary for CSV storage.

**Key distinction:** Saving a class definition in a `.py` file preserves the *code*. It does not automatically save objects created from that class.

In [1]:
from pathlib import Path

module_code = """from dataclasses import dataclass

@dataclass
class LaundryOrder:
    order_id: int
    customer: str
    pounds: float
    price_per_pound: float = 2.50

    def __post_init__(self):
        if not self.customer.strip():
            raise ValueError("Customer name cannot be blank")
        if self.pounds <= 0:
            raise ValueError("Pounds must be greater than zero")

    def total(self) -> float:
        return round(self.pounds * self.price_per_pound, 2)

    def to_record(self) -> dict:
        return {
            "order_id": self.order_id,
            "customer": self.customer,
            "pounds": self.pounds,
            "price_per_pound": self.price_per_pound,
            "total": self.total(),
        }
"""
Path("laundry_order.py").write_text(module_code, encoding="utf-8")
print("Saved laundry_order.py")

Saved laundry_order.py


In [2]:
from laundry_order import LaundryOrder

sample = LaundryOrder(101, "Mary", 12)
print(sample)
print("Total:", sample.total())
print("CSV-ready record:", sample.to_record())

LaundryOrder(order_id=101, customer='Mary', pounds=12, price_per_pound=2.5)
Total: 30.0
CSV-ready record: {'order_id': 101, 'customer': 'Mary', 'pounds': 12, 'price_per_pound': 2.5, 'total': 30.0}


**Discuss:** Why is the calculation inside the class instead of repeated in every application? What happens if the price changes?  
-->The calculation is inside the class so that different applications can reuse the same logic without having to write it again. This also makes the code easier to maintain and reduces errors. If the price changes, we can update the default price in the class instead of changing it in every application that uses it.

## Part 2 — Application A: Order Entry (15 minutes)

Save another `.py` file. This application **imports** `LaundryOrder` rather than copying the class definition. It appends records to `laundry_orders.csv`. A header is written only when the file is new or empty.

In [3]:
entry_code = """import csv
from pathlib import Path
from laundry_order import LaundryOrder

CSV_PATH = Path("laundry_orders.csv")
FIELDS = ["order_id", "customer", "pounds", "price_per_pound", "total"]

def save_order(order: LaundryOrder, path: Path = CSV_PATH) -> None:
    needs_header = not path.exists() or path.stat().st_size == 0
    with path.open("a", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=FIELDS)
        if needs_header:
            writer.writeheader()
        writer.writerow(order.to_record())

if __name__ == "__main__":
    save_order(LaundryOrder(101, "Mary", 12))
    save_order(LaundryOrder(102, "James", 8))
    print("Two orders saved to", CSV_PATH)
"""
Path("order_entry_app.py").write_text(entry_code, encoding="utf-8")
print("Saved order_entry_app.py")

Saved order_entry_app.py


Run the order-entry application. For a repeatable demonstration, the next cell deletes the *lab's demo CSV* first. **Do not use that reset step on real customer data.**

In [4]:
from pathlib import Path
import subprocess, sys

csv_path = Path("laundry_orders.csv")
if csv_path.exists():
    csv_path.unlink()  # DEMO RESET ONLY
subprocess.run([sys.executable, "order_entry_app.py"], check=True)

CompletedProcess(args=['/usr/bin/python3', 'order_entry_app.py'], returncode=0)

In [12]:
print(Path("laundry_orders.csv").read_text(encoding="utf-8"))

order_id,customer,pounds,price_per_pound,total
101,Mary,12,2.5,30.0
102,James,8,2.5,20.0
103,Ankit Aryal,18,2.5,45.0



## Part 3 — Application B: Reporting (15 minutes)

This is a **different application**. It imports the **same** `LaundryOrder` class and reads the CSV file. CSV values are strings, so we convert numbers back to `int` and `float` before rebuilding each object.

In [10]:
report_code = """import csv
from pathlib import Path
from laundry_order import LaundryOrder

CSV_PATH = Path("laundry_orders.csv")

def load_orders(path: Path = CSV_PATH) -> list[LaundryOrder]:
    if not path.exists():
        return []
    with path.open("r", newline="", encoding="utf-8") as file:
        return [
            LaundryOrder(
                order_id=int(row["order_id"]),
                customer=row["customer"],
                pounds=float(row["pounds"]),
                price_per_pound=float(row["price_per_pound"]),
            )
            for row in csv.DictReader(file)
        ]

if __name__ == "__main__":
    orders = load_orders()
    for order in orders:
        print(f"#{order.order_id}: {order.customer} — ${order.total():.2f}")
    print(f"Orders: {len(orders)} | Revenue: ${sum(o.total() for o in orders):.2f}")

    if orders:
        highest = max(orders, key=lambda order: order.total())
        print(f"Highest-value order: #{highest.order_id}: {highest.customer} — ${highest.total():.2f}")

"""
Path("order_report_app.py").write_text(report_code, encoding="utf-8")
print("Saved order_report_app.py")

Saved order_report_app.py


In [13]:
result = subprocess.run(
    [sys.executable, "order_report_app.py"],
    check=True,
    capture_output=True,
    text=True
)

print(result.stdout)

#101: Mary — $30.00
#102: James — $20.00
#103: Ankit Aryal — $45.00
Orders: 3 | Revenue: $95.00
Highest-value order: #103: Ankit Aryal — $45.00



## Part 4 — Prove persistence (5 minutes)

The two applications ran in **separate Python processes**, yet Application B could read orders saved by Application A. The data is on disk, not just in a variable.

**Optional restart demonstration:** Restart the notebook kernel, then run the cell below *without rerunning the demo-reset cell*. You should still see the orders, provided the files remain in the working folder.

In [14]:
from order_report_app import load_orders
orders_after_restart = load_orders()
print("Recovered", len(orders_after_restart), "orders from CSV")
for order in orders_after_restart:
    print(order.customer, order.total())

Recovered 3 orders from CSV
Mary 30.0
James 20.0
Ankit Aryal 45.0


## Student challenge (homework, due Monday beginning of class)


1. Add a code cell below this markdown block.
2. Add a new order for a customer (use your name) using `save_order` from `order_entry_app.py`.
3. Modify the reporting app to show the highest-value order.
4. Rerun the report
5. Make sure your name and the date are in the top markdown cell.


**Exit ticket:** In 2–3 sentences, explain the difference between (a) saving a class in a `.py` file, (b) creating an object in memory, and (c) saving object data to a CSV file.

**Important limitations:** CSV is fine for a classroom demo, but it does not enforce unique order IDs, prevent simultaneous-write conflicts, or provide database transactions. A production application should generally use a database.

In [9]:

from laundry_order import LaundryOrder
from order_entry_app import save_order

my_order = LaundryOrder(103, "Ankit Aryal", 18)
save_order(my_order)

print("My order was saved successfully!")


My order was saved successfully!


Exit ticket: Saving a class in a `.py` file stores the reusable Python code that defines how objects should behave. Creating an object in memory creates an instance of that class while the program is running. Saving the object's data to a CSV file allows the information to be recovered even after the program has stopped.